# Лекция 12 — Работа с интернет-страницами: задачи для самостоятельной работы

**Темы:** HTML и HTTP (запрос, ответ, заголовки, коды состояния), URL и его разбор (`urllib.parse`), `urlencode` параметров, форматы CSV и JSON (`csv`, `json`), `base64`, парсинг HTML через BeautifulSoup (`find`, `find_all`, CSS-селекторы, `get_text`, атрибуты), ответы API и их обработка, `requests` (`Response`, `text`, `content`, `params`) — и его имитация на локальных данных.

- **Задачи 1–10** — простые
- **Задачи 11–20** — средние
- **Задачи 21–30** — сложные

Все решения выполняются **без интернета**: HTML лежит в строковых литералах, ответы API — обычные словари и JSON-строки, файлы создаются во временной директории и удаляются. Реальные HTTP-запросы в задачах не нужны.


## Простые задачи


### 1. Найти тег и его текст
Дана HTML-строка (это «скачанная» страница, интернет не нужен). Соберите `BeautifulSoup` и найдите заголовок `h1`: его текст и значение атрибута `id`. Посчитайте число абзацев `p`.

```python
html = (
    "<html><body>"
    "<h1 id='title'>Погода в Москве</h1>"
    "<p class='today'>Сегодня +18</p>"
    "<p class='tomorrow'>Завтра +21</p>"
    "<p class='note'>Данные примера, не интернет</p>"
    "</body></html>"
)
```

Проверьте: `title == "Погода в Москве"`, `n_p == 3`, `soup.find("h4") is None`.

**Подсказка:** `soup = BeautifulSoup(html, "html.parser")`, затем `soup.find("h1").get_text()` и `len(soup.find_all("p"))`; отсутствующий тег возвращается как `None`.


### 2. Все теги и их атрибуты
Найдите все ссылки `a`, соберите список их `href` (у части ссылок атрибута `href` нет) и отдельно — список «внешних» адресов, начинающихся с `http`.

```python
html = (
    "<ul>"
    "<li><a href='https://example.com/a'>A</a></li>"
    "<li><a href='/local/b.html'>B</a></li>"
    "<li><a href='http://example.org/c'>C</a></li>"
    "<li><a href='#top'>Наверх</a></li>"
    "<li><a>Без адреса</a></li>"
    "</ul>"
)
```

Проверьте: ссылок 5, список `href` совпадает с ожидаемым, внешних ровно 2.

**Подсказка:** `soup.find_all("a")`, а значение атрибута берите через `tag.get("href", "")`, чтобы не поймать `KeyError`.


### 3. get_text: strip и separator
Из карточки товара получите: «сырой» текст со всем мусором, «чистый» текст с разделителем `|` и список отдельных слов.

```python
html = (
    "<div class='card'>"
    "<h2>Товар</h2>"
    "<p>  Описание  в  двух  строках  </p>"
    "<ul><li>дешёвый</li><li>хороший</li></ul>"
    "</div>"
)
```

Проверьте: `div.get_text(strip=True, separator="|") == "Товар|Описание  в  двух  строках|дешёвый|хороший"`, а `get_text().split()` даёт **6** слов, причём последнее — склеенное `дешёвыйхороший`; с `separator=" "` слов становится 7.

**Подсказка:** `strip=True` убирает пробелы по краям каждого куска текста (двойные пробелы внутри остаются), но соседние теги без пробела между ними склеиваются — поэтому и нужен `separator`; искать div по классу удобно через `soup.find("div", class_="card")`.


### 4. Атрибуты тега как словарь
У тега `img` возьдите атрибуты целиком как словарь, отдельно `src` и `alt`, а отсутствующий `title` — безопасно.

```python
html = "<div id='gallery'><img src='img/cat.jpg' alt='Кот' width='200'></div>"
```

Проверьте: `attrs == {"src": "img/cat.jpg", "alt": "Кот", "width": "200"}`, `img.get("title") is None`, `"alt" in img.attrs`, у `div` атрибут `id == "gallery"`.

**Подсказка:** у любого тега есть словарь `.attrs`; все его значения — строки, даже `width='200'`.


### 5. Разбор URL на части
Разложите адрес страницы на составляющие и проверьте каждую: протокол, хост с портом, путь, строку параметров, фрагмент.

```python
url = "https://www.example.com:443/path/to/page.html?key=value&lang=ru#bottom"
```

Проверьте: `scheme == "https"`, `netloc == "www.example.com:443"`, `path == "/path/to/page.html"`, `query == "key=value&lang=ru"`, `fragment == "bottom"`.

**Подсказка:** `from urllib.parse import urlparse` — она возвращает именованный кортеж; адрес без протокола (`www.example.com/a`) разбирается «криво»: `scheme` пустой, а всё значение уходит в `path`.


### 6. Сборка URL: urlunparse и urljoin
Из кортежа частей соберите адрес обратно, убедитесь, что `urlparse` → `urlunparse` — обратимая операция, и разрешите относительные ссылки от адреса страницы.

```python
parts = ("https", "docs.example.ru", "/articles/2024/python.html", "", "page=2", "top")
base = "https://site.ru/a/b/index.html"
```

Проверьте: собранная строка равна `https://docs.example.ru/articles/2024/python.html?page=2#top`, `urljoin(base, "../c/d.html") == "https://site.ru/a/c/d.html"`, `urljoin(base, "img/logo.png") == "https://site.ru/a/b/img/logo.png"`.

**Подсказка:** порядок полей у `urlunparse` такой же, как у `urlparse`: scheme, netloc, path, params, query, fragment.


### 7. urlencode параметров запроса
Из словаря параметров соберите query-строку так, чтобы порядок параметров был детерминированным, и проверьте обратное разбирать.

```python
params = {"q": "погода в Москве", "lang": "ru", "days": 3}
```

Проверьте: `query == "days=3&lang=ru&q=%D0%BF%D0%BE%D0%B3%D0%BE%D0%B4%D0%B0+%D0%B2+%D0%9C%D0%BE%D1%81%D0%BA%D0%B2%D0%B5"`, а `parse_qs(query)["q"] == ["погода в Москве"]`.

**Подсказка:** `urlencode(sorted(params.items()))` фиксирует порядок по ключам; кириллица и пробелы экранируются сами (`quote_plus`: пробел становится `+`).


### 8. json.loads: из строки в объект Python
Превратите JSON-ответ в объект Python и проверьте, как JSON-типы превратились в питоновские.

```python
raw = '{"город": "Казань", "температура": 19.5, "ясно": true, "ветер": null, "прогнозы": [1, 2, 3], "count": 7}'
```

Проверьте: `data["ясно"] is True`, `data["ветер"] is None`, `температура` — `float`, `count` — `int`, `прогнозы == [1, 2, 3]`, в словаре 6 ключей.

**Подсказка:** `json.loads` знает только JSON-типы: `true/false/null` становятся `True/False/None`, а ключи — всегда строками.


### 9. json.dumps: из объекта в строку
Соберите JSON-строку трижды: компактную с кириллицей и сортировкой ключей, «машинную» с экранированной кириллицей и читаемую с отступами.

```python
obj = {"город": "Москва", "temp": 18.5, "ok": True, "note": None}
```

Проверьте: компактная строка равна `'{"note": null, "ok": true, "temp": 18.5, "город": "Москва"}'`, в обычном `json.dumps(obj)` встречается `\\u0433`, а в читаемой есть переносы строк и отступы.

**Подсказка:** `json.dumps(obj, ensure_ascii=False, sort_keys=True)`; ключи сортируются по кодам символов, поэтому кириллица оказывается после латиницы.


### 10. base64: декодирование «куска ответа»
В логах лежит base64-запись текста из ответа сервера. Декодируйте её и проверьте, что кодирование обратимо.

```python
encoded = "0J/RgNC40LLQtdGCLCDQvNC40YAh"
```

Проверьте: декодированная строка равна `Привет, мир!`, повторное кодирование даёт ровно ту же строку, длина base64 кратна 4.

**Подсказка:** `base64.b64decode(encoded).decode("utf-8")` — сначала байты, потом текст. Base64 — это кодирование для безопасной пересылки байтов, а не шифрование: расшифровать может любой.


## Средние задачи


### 11. HTML-таблица → pandas DataFrame
Спарсите таблицу со страницы и превратите её в `DataFrame`: сначала вручную (списки списков), затем вторым способом через `io.StringIO` и модуль `csv`. Числовой столбец приведите к `float`.

```python
html = (
    "<table id='stocks'>"
    "<tr><th>Товар</th><th>Склад</th><th>Цена</th></tr>"
    "<tr><td>Молоко</td><td>Восток</td><td>89.5</td></tr>"
    "<tr><td>Хлеб</td><td>Запад</td><td>42.0</td></tr>"
    "<tr><td>Сыр</td><td>Восток</td><td>310.25</td></tr>"
    "<tr><td>Масло</td><td>Центр</td><td>159.9</td></tr>"
    "</table>"
)
```

Проверьте: `shape == (4, 3)`, имена столбцов — `Товар, Склад, Цена`, `mean` цены равен `150.41` (округлите), оба способа дали одинаковую форму таблицы.

**Подсказка:** первая строка — `th`, остальные — `tr` со вложенными `td`; `find_all("tr")` и срез `[1:]`.


### 12. Выборка по CSS-селекторам
По разметке каталога фильмов выберите элементы разными селекторами: прямой потомок, несколько классов, класс + псевдокласс `:not`, вложенность.

```python
html = (
    "<div class='films'>"
    "<div class='film'><h2 class='title'>Титаник</h2><span class='rating high'>9.5</span></div>"
    "<div class='film'><h2 class='title'>Аватар</h2><span class='rating low'>7.8</span></div>"
    "<div class='film hidden'><h2 class='title'>Черновик</h2><span class='rating low'>3.0</span></div>"
    "<aside><h2 class='title'>Реклама</h2></aside>"
    "</div>"
)
```

Проверьте: `div.film > h2.title` даёт 3 заголовка, `span.rating.high` — один (`9.5`), `div.film:not(.hidden) > h2` — два (Титаник, Аватар), `select_one("aside h2")` — Реклама, всего `.rating` — 3.

**Подсказка:** `soup.select("...")` возвращает список, `soup.select_one("...")` — первый элемент или `None`; класс `film hidden` ищется как `.film.hidden`.


### 13. Все ссылки → дерево путей
Соберите со страницы все ссылки, переведите относительные в абсолютные (`urljoin` от адреса страницы), оставьте только тот же домен и сгруппируйте по первому сегменту пути.

```python
BASE = "https://wiki.example.ru/guide/start.html"
html = (
    "<div>"
    "<a href='../articles/python/index.html'>Python</a>"
    "<a href='../articles/python/list.html'>Списки</a>"
    "<a href='../articles/sql.html'>SQL</a>"
    "<a href='/guide/next.html'>Далее</a>"
    "<a href='https://other.org/x.html'>Внешний</a>"
    "<a href='mailto:hi@example.ru'>Почта</a>"
    "</div>"
)
```

Проверьте: внутренних ссылок 4, ключи дерева — `articles` и `guide`, в `articles` 3 ссылки, в `guide` 1.

**Подсказка:** отфильтруйте по `urlparse(u).netloc == "wiki.example.ru"` и схеме `http/https`, затем `path.strip("/").split("/")[0]` и `dict.setdefault(key, []).append(...)`.


### 14. Query-строка API: parse_qs и «мысленный» запрос
Дана строка параметров реального API-запроса. Разберите её, посмотрите, куда деваются повторяющиеся ключи, измените город, поменяйте лимит и соберите новый URL.

```python
query = "city=%D0%9C%D0%BE%D1%81%D0%BA%D0%B2%D0%B0&units=metric&tags=xml&tags=json&limit=10"
api = "https://api.example.ru/v1/forecast"
```

Проверьте: `parse_qs(query)["city"] == ["Москва"]`, `tags == ["xml", "json"]`, `dict(parse_qsl(query))["tags"] == "json"` (повтор потерян), а итоговый URL начинается с `api + "?"` и `parse_qs` из него возвращает новый город.

**Подсказка:** `parse_qs` всегда отдаёт списки значений (а `parse_qsl` — список пар, и в `dict` от него остаётся последнее значение); пересоберите строку через `urlencode({"city": ..., "units": ..., "limit": 50})` — он сохраняет порядок ключей словаря.


### 15. JSON-ответ API со вложенными списками → DataFrame
Ответ магазина: под ключом `results` лежит список объектов, у части из них поле `tags` — список. Соберите `DataFrame` из этого списка (без `json_normalize`), список-столбец превратите в строку.

```python
raw = (
    '{"meta": {"total": 4, "page": 1},'
    ' "results": ['
    '{"id": 1, "name": "Тетрадь", "tags": ["канц", "школа"], "price": 59.9, "stock": true},'
    '{"id": 2, "name": "Ручка", "tags": ["канц"], "price": 35.0, "stock": true},'
    '{"id": 3, "name": "Рюкзак", "tags": ["школа", "спорт"], "price": 1990.0, "stock": false},'
    '{"id": 4, "name": "Линейка", "tags": [], "price": 25.0, "stock": true}'
    ']}'
)
```

Проверьте: `shape == (4, 5)`, столбцы идут в исходном порядке, `tags` в нулевой строке равен `канц,школа`, у линейки — пустая строка, сумма `price` равна `2109.9`, в наличии 3 товара.

**Подсказка:** `pd.DataFrame(json.loads(raw)["results"])` из списка словарей сам строит столбцы; для `tags` используйте `.apply(lambda t: ",".join(t))`.


### 16. Запись собранного парсинга в CSV
Спарсите список навыков со страницы и запишите результат в CSV во временный файл (`tempfile.mkstemp`), затем перечитайте и сравните. Файл обязательно удалите.

```python
items = ["python", "sql", "git", "bash"]
html = "<ul>" + "".join(
    "<li data-skill='%s' data-level='%d'>%s</li>" % (name, i % 3 + 1, name)
    for i, name in enumerate(items)
) + "</ul>"
```

Проверьте: файл начинается с строки `навык,уровень`, перечитанные записи равны `[{"навык": "python", "уровень": "1"}, {"навык": "sql", "уровень": "2"}, {"навык": "git", "уровень": "3"}, {"навык": "bash", "уровень": "1"}]`, после `os.remove` файл отсутствует.

**Подсказка:** `csv.DictWriter(f, fieldnames=[...])` + `writeheader()` + `writerows(rows)`; файл открывайте с `encoding="utf-8", newline=""`; удаляйте в блоке `finally`.


### 17. HTTP-заголовки и коды состояния
Реализуйте класс-заглушку ответа `FakeResponse` (поля `url`, `status_code`, `headers`, `text`) с методами: регистронезависимый поиск заголовка, `raise_for_status()`. Проверьте поведение на четырёх «ответах».

```python
class FakeResponse:
    def __init__(self, url, status_code, headers, text=""):
        self.url = url
        self.status_code = status_code
        self.headers = headers
        self.text = text
```

Ответы: `200` с `Content-Type: text/html; charset=utf-8`, `301` с `Location`, `404`, `500`.

Проверьте: заголовок найден независимо от регистра, отсутствующий заголовок даёт `None`, `Location` ведёт на новый адрес, `raise_for_status()` не бросает для `2xx/3xx` и бросает `RuntimeError` для `404` и `500` (в тексте кода статуса).

**Подсказка:** сравнивайте `key.lower() == name.lower()` в цикле по `headers.items()`; классы статусов — `status_code // 100`.


### 18. Битые теги: два парсера, два результата
Страница «сгенерирована дрожащими руками»: незакрытые `p` и `li`, лишняя закрывающая `</div>`, перепутанные `b`/`i`. Разберите её дважды — наивным `html.parser` и парсером `lxml` — и сравните полученное дерево.

```python
html = (
    "<div id='content'>"
    "<p>Первый абзац без закрытия"
    "<p>Второй абзац"
    "<ul><li>один<li>два</ul>"
    "</div>"
    "</div>"
    "<b><i>жирный курсив</b></i>"
    "<p>После мусора"
)
```

Проверьте: оба парсера находят 3 абзаца и не падают; у `html.parser` структура «поехала» (текст первого `p` глотает вложенное, а `ul` оказывается внутри `p`); `lxml` чинит разметку как браузер: тексты абзацев `['Первый абзац без закрытия', 'Второй абзац', 'После мусора']`, `parent.name` у `li` — `ul`, у `ul` — `div`, у `div#content` непосредственные дети `['p', 'p', 'ul']`, текст `b` восстановлен.

**Подсказка:** `BeautifulSoup(html, "html.parser")` и `BeautifulSoup(html, "lxml")`; смотрите `.parent.name` и обход `.children`, а не надейтесь на исходную вложенность тегов.


### 19. Числа из «человеческого» текста
Цены в разметке написаны для людей: неразрывные пробелы, запятая как десятичный разделитель, символы валют. Соберите список чисел и посчитайте сводку.

```python
html = (
    "<ul class='prices'>"
    "<li>Молоко: 89\xa0₽</li>"
    "<li>Корм: 1\xa0299 ₽</li>"
    "<li>Носки: $12.50</li>"
    "<li>Велосипед: 3 499,90 грн</li>"
    "</ul>"
)
```

Проверьте: список цен равен `[89.0, 1299.0, 12.5, 3499.9]`, сумма равна `4900.4`, максимум `3499.9`, список названий — `["Молоко", "Корм", "Носки", "Велосипед"]`.

**Подсказка:** очистите строку (`replace("\xa0", "")`, `replace(" ", "")`, `replace(",", ".")`), а число возьмите регуляркой `re.search(r"[-+]?\d+(?:\.\d+)?", ...)`.


### 20. Карточки товаров → DataFrame со сводкой
По классам выберите карточки, из каждой — название, цену и категорию из `data-*` атрибута; соберите `DataFrame`, отсортируйте и посчитайте сводку.

```python
html = (
    "<div class='catalog'>"
    "<div class='card' data-cat='книги'><h3 class='name'>Python за 30 дней</h3><span class='price'>1 290 ₽</span></div>"
    "<div class='card' data-cat='книги'><h3 class='name'>Чистый код</h3><span class='price'>2 150 ₽</span></div>"
    "<div class='card' data-cat='гаджеты'><h3 class='name'>Мышь</h3><span class='price'> 990 ₽ </span></div>"
    "<div class='card' data-cat='гаджеты'><h3 class='name'>Клавиатура</h3><span class='price'>3 490 ₽</span></div>"
    "</div>"
)
```

Проверьте: `shape == (4, 3)`, порядок названий после сортировки по убыванию цены — `Клавиатура, Чистый код, Python за 30 дней, Мышь`, сумма цен `7920.0`, в каждой категории по 2 товара.

**Подсказка:** внутри карточки ищите через `card.select_one("h3.name")`, категорию берите как `card["data-cat"]`, из цены уберите пробелы и `₽`.


## Сложные задачи


### 21. Мини-краулер по локальным страницам
«Сайт» лежит в словаре `PAGES` (путь → HTML). Реализуйте обход в глубину от `/`: посещать страницу один раз, идти только по внутренним ссылкам (тот же домен, путь начинается с `/`), соблюдать лимит `max_pages`. Циклы в графе обязательны.

```python
PAGES = {
    "/": '<a href="/a">a</a><a href="/b">b</a>',
    "/a": '<a href="/a1">a1</a><a href="/">root</a>',
    "/b": '<a href="/b1">b1</a><a href="/a">a</a><a href="https://other.org/x">out</a>',
    "/a1": '<a href="/b1">to b1</a>',
    "/b1": "<p>лист</p>",
    "/orphan": "<a href='/'>в начало</a>",
}
```

Проверьте: из `/` достижимо ровно 5 страниц (`{"/", "/a", "/b", "/a1", "/b1"}`), `/orphan` не посещена, первая страница — `/`, а `crawl("/", max_pages=2)` возвращает 2 адреса.

**Подсказка:** стек + множество `seen`; ссылка «уже видена» — не толкать в стек; останавливаться, когда `len(order) == max_pages`.


### 22. Свой mock-транспорт для requests
Реализуйте «HTTP-клиент без интернета»: класс `FakeResponse` с атрибутами `url`, `status_code`, `headers`, `text` и методом `json()`, а также функцию `get(url, params=None)`, которая берёт данные из локального словаря-«базы». Поверх напишите `api_get(url)` с проверкой статуса и своим классом `ApiError`.

```python
DB = {
    "/users": [{"id": 1, "name": "Анна"}, {"id": 2, "name": "Борис"}],
    "/users/1": {"id": 1, "name": "Анна", "roles": ["admin"]},
    "/users/2": {"id": 2, "name": "Борис", "roles": []},
}
```

Проверьте: `api_get("https://api.local/users")` отдаёт имена `["Анна", "Борис"]`, `users/1` содержит роль `admin`, а запрос `users/99` бросает `ApiError` с текстом `404`.

**Подсказка:** путь «URL» получайте через `urlparse(url).path`; параметры при желании доклеивайте `urlencode(sorted(params.items()))` — база всё равно ищется по пути.


### 23. Пагинация API
API отдаёт по 10 объектов и указывает следующую страницу в поле `next` (`None` — конец; последняя страница неполная). Обойдите все страницы, соберите данные в один список, посчитайте число обращений и соберите `DataFrame` (без `json_normalize`).

```python
PAGES = {
    1: {"items": [{"id": i} for i in range(1, 11)], "next": "/items?page=2"},
    2: {"items": [{"id": i} for i in range(11, 21)], "next": "/items?page=3"},
    3: {"items": [{"id": i} for i in range(21, 26)], "next": None},
}
```

Проверьте: собрано 25 записей, идентификаторы уникальны (25 штук), сделано ровно 3 обращения, первые три `id` — `1, 2, 3`.

**Подсказка:** цикл `while url is not None`; страницу доставайте из `PAGES` по номеру из `parse_qs(urlparse(url).query)`; адрес следующей страницы — `HOST + next`.


### 24. Дедупликация и очистка спарсенных данных
«Спаутили» три страницы и получили мусор: дубликаты с разным регистром, лишние пробелы, пустые значения. Приведите данные в порядок и посчитайте сводку.

```python
RAW = [
    {"город": " Москва ", "канал": "Telegram"},
    {"город": "москва", "канал": "telegram"},
    {"город": "Москва", "канал": None},
    {"город": " СПб", "канал": "Telegram"},
    {"город": "СПб ", "канал": "YouTube"},
    {"город": "Казань", "канал": None},
    {"город": "казань", "канал": "YouTube"},
]
```

Проверьте: было 7 строк, после нормализации и `drop_duplicates` — 6, без пустых `канал` — 4, сводка по городам `{"москва": 1, "спб": 2, "казань": 1}`, уникальных каналов 2.

**Подсказка:** нормализация — `.str.strip().str.lower()` (пустые значения останутся `NaN`), затем `drop_duplicates()`, `dropna(subset=["канал"])` и `groupby("город").size()`.


### 25. JSON-API курсов валют и конвертер
Ответ валютного API: базовая валюта, плоский словарь `rates` и вложенная `meta`. Соберите конвертер: пересчёт любой пары валют через базовую, округление до копеек, понятная ошибка на неизвестную валюту.

```python
raw = (
    '{"base": "RUB", "date": "2026-10-20",'
    ' "rates": {"USD": 0.0109, "EUR": 0.0104, "CNY": 0.0775, "KZT": 5.83},'
    ' "meta": {"source": "example-api", "updated": {"by": "bot", "lag_min": 5}}}'
)
```

Проверьте: `meta["updated"]["lag_min"] == 5`, `convert(100, "USD", "EUR") == 95.41`, `convert(500, "RUB", "USD") == 5.45`, `convert(1000, "KZT", "RUB") == 171.53`, `convert(77, "RUB", "RUB") == 77.0`, а `convert(1, "GBP", "RUB")` бросает `KeyError`.

**Подсказка:** сначала переведите сумму в базовую валюту (`amount / rates[cur]`, для базовой — без деления), затем умножьте на `rates[to]`; в конце `round(..., 2)`.


### 26. Парсинг HTML-формы и «отправка» GET-запроса
Найдите форму, соберите её поля (включая выбранную опцию `select` и пустые `input`), «отправьте» GET-запрос — то есть соберите итоговый URL — и проверьте байты адреса.

```python
html = (
    "<form action='https://shop.example.ru/search' method='get'>"
    "<input type='text' name='q' value='курсы python'>"
    "<input type='hidden' name='src' value='site'>"
    "<select name='cat'>"
    "<option value='all'>Все</option>"
    "<option value='ml' selected>ML</option>"
    "</select>"
    "<input type='text' name='empty'>"
    "</form>"
)
```

Проверьте: метод — `get`, поля `{"q": "курсы python", "src": "site", "cat": "ml"}` (пустое поле отброшено), итоговый URL равен `https://shop.example.ru/search?cat=ml&q=%D0%BA%D1%83%D1%80%D1%81%D1%8B+python&src=site`, `url.encode("ascii")` не падает, в адресе 10 символов `%` (по два на каждую кириллическую букву).

**Подсказка:** `form.find_all(["input", "select", "textarea"])`; выбранную опцию ищите через `el.find("option", selected=True)` и не забывайте `el.get("value", "")`.


### 27. Сжатие и кодирование: «gzip-ответ» без сети
Серверы часто отдают тело сжатым (`Content-Encoding: gzip`). Локально сожмите HTML, оберните байты в класс `GzResponse`, чей атрибут `text` распаковывает и декодирует тело, и разберите полученную разметку.

```python
import gzip

HTML = "<table>" + "<tr><td>Ячейка</td></tr>" * 200 + "</table>"
packed = gzip.compress(HTML.encode("utf-8"))
```

Проверьте: `packed[:2] == b"\x1f\x8b"` (магические байты gzip), сжатые данные короче исходных, `GzResponse(packed).text == HTML`, в распакованной разметке 200 ячеек `td`, gzip от json-строки распаковывается в тот же объект, а «несжатый» ответ (`Content-Encoding: identity`) читается напрямую.

**Подсказка:** `gzip.decompress(resp.content).decode("utf-8")`; в реальном `requests` это делает `response.text`, если заголовок `Content-Encoding` говорит о gzip.


### 28. Rate-limit: token bucket
Чтобы не превращать парсер в атаку, сделайте ограничитель «ведро токенов»: ёмкость `capacity`, скорость `rate` токенов в секунду, часы инъектируются (никаких `time.sleep` — время сдвигаем вручную).

```python
class FakeClock:
    def __init__(self, now=0.0):
        self.now = now
    def __call__(self):
        return self.now
    def advance(self, dt):
        self.now += dt
```

Проверьте: при `capacity=3, rate=1.0` из пяти мгновенных запросов проходят 3; после `advance(2.0)` проходят ещё ровно 2 и затем снова отказ; «ожидание» 10 запросов при `capacity=2, rate=0.5` даёт времена `[0.0, 0.0, 2.0, 4.0, ..., 16.0]`.

**Подсказка:** при каждом обращении доливайте `tokens = min(capacity, tokens + (now - last) * rate)` и тратьте 1 токен; в симуляции крутите `while not bucket.allow(): clock.advance(1.0)`.


### 29. Retry-декоратор для падающих запросов
Запросы к API падают изредка. Напишите декоратор `retry(times, delay)`: повтор после `ConnectionError`, экспоненциальная задержка (`delay`, `2*delay`, `4*delay`), записанные в атрибут `waits`, и финальный бросок исходной ошибки, когда попытки кончились.

```python
def make_flaky(n_fail):
    state = {"calls": 0}
    def flaky():
        state["calls"] += 1
        if state["calls"] <= n_fail:
            raise ConnectionError("сбой сети, попытка %d" % state["calls"])
        return "страница %d" % state["calls"]
    flaky.state = state
    return flaky
```

Проверьте: функция, падающая 2 раза, с `retry(times=3, delay=0.1)` возвращает `"страница 3"`, сделав 3 вызова, `waits == [0.1, 0.2]`; функция, падающая 5 раз, с `retry(times=2)` бросает `ConnectionError` с текстом `попытка 3`, сделав ровно 3 вызова.

**Подсказка:** `functools.wraps`, цикл `for attempt in range(times + 1)`, `try/except ConnectionError`, задержки копить в списке замыкания и вешать на обёртку (`wrapper.waits = waits`).


### 30. Мини-пайплайн: скачать → распарсить → очистить → CSV → перечитать
Соберите весь конвейер: «скачайте» страницу через mock-ответ, выберите карточки прогноза, почистите значения (регистр, пробелы, `н/д` вместо числа), сохраните CSV во временный файл и перечитайте его, получив итоговую сводку.

```python
class FakeResp:
    def __init__(self, text, status_code=200):
        self.status_code = status_code
        self.headers = {"Content-Type": "text/html; charset=utf-8"}
        self.text = text

PAGE = (
    "<html><body><div class='rows'>"
    "<div class='row'><span class='city'> Москва </span><span class='temp'>-2.5</span><span class='date'>2026-10-01</span></div>"
    "<div class='row'><span class='city'>москва</span><span class='temp'> 1.0 </span><span class='date'>2026-10-02</span></div>"
    "<div class='row'><span class='city'>Тверь</span><span class='temp'>н/д</span><span class='date'>2026-10-01</span></div>"
    "<div class='row'><span class='city'> Тверь </span><span class='temp'>3.5</span><span class='date'>2026-10-02</span></div>"
    "<div class='row'><span class='city'>Казань</span></div>"
    "</div></body></html>"
)
```

Проверьте: в «сырых» записях 5 строк, в чистом `DataFrame` 3 строки (две битые отсеяны), CSV перечитывается с теми же столбцами, среднее по колонке `температура` равно `0.67`, сводка по городам — `{"москва": -0.75, "тверь": 3.5}`, файл удалён.

**Подсказка:** пропуск строки — если нет `.city`; `float("н/д")` ловите в `try/except ValueError` (значение `None`), дубликаты снимите `drop_duplicates(subset=["город", "дата"])`, а пустые температуры — `dropna(subset=["температура"])`.


## Где потренироваться дальше

- [Документация BeautifulSoup 4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [requests: Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [urllib.parse — разбор и сборка URL](https://docs.python.org/3/library/urllib.parse.html)
- [json — чтение и запись JSON](https://docs.python.org/3/library/json.html)
- [csv — табличный формат](https://docs.python.org/3/library/csv.html)
- [base64 — кодирование байтов](https://docs.python.org/3/library/base64.html)
- [MDN: обзор HTML](https://developer.mozilla.org/ru/docs/Web/HTML)
- [MDN: обзор HTTP](https://developer.mozilla.org/ru/docs/Web/HTTP)
- [httpbin — песочница для проверки HTTP-запросов](https://httpbin.org/)
